In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ═══════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════
import os, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2
from scipy.fftpack import dct as scipy_dct
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix, roc_curve)
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
import joblib
import tensorflow as tf
from keras import layers, models
from tensorflow.keras.utils import to_categorical
from scipy.ndimage import uniform_filter

np.random.seed(42)
random.seed(42)
print("✓ Imports OK")

# ═══════════════════════════════════════════════════════
# CHEMINS
# ═══════════════════════════════════════════════════════
DETECTION_PATH   = '/kaggle/input/datasets/marcozuppelli/stegoimagesdataset'
CLASSIF_PATH     = '/kaggle/input/datasets/diegozanchett/digital-steganography'
CLEAN_PATH       = f'{DETECTION_PATH}/train/train/clean'

EXTENSIONS   = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff'}
CLASS_NAMES  = ['dct', 'fft', 'lsb', 'lsb_grayscale', 'ssb4', 'ssbn']

print(f"✓ Chemins configurés")

# ═══════════════════════════════════════════════════════
# CHARGEMENT DATASET 1 (détection binaire)
# ═══════════════════════════════════════════════════════
def load_detection_dataset(root):
    root = Path(root)
    splits = {}
    for split in ['train', 'val', 'test']:
        for candidate in [root/split/split, root/split]:
            if candidate.exists() and candidate.is_dir():
                split_path = candidate
                break
        rows = []
        for folder in sorted(split_path.iterdir()):
            if not folder.is_dir():
                continue
            name = folder.name.lower()
            if name == 'clean':
                label = 0
            elif name in {'stego', 'stego_b64', 'stego_zip'}:
                label = 1
            else:
                continue
            for p in sorted(folder.rglob('*')):
                if p.suffix.lower() in EXTENSIONS:
                    rows.append({'path': str(p), 'label': label,
                                 'split': split})
        splits[split] = pd.DataFrame(rows)
        c = (splits[split]['label']==0).sum()
        s = (splits[split]['label']==1).sum()
        print(f"  {split:6s} → {len(splits[split])} images (clean={c}, stego={s})")
    return splits['train'], splits['val'], splits['test']

print("\n=== Dataset 1 — Détection binaire ===")
tr1, va1, te1 = load_detection_dataset(DETECTION_PATH)

# ═══════════════════════════════════════════════════════
# CHARGEMENT DATASET 2 (classification technique)
# ═══════════════════════════════════════════════════════
def load_classification_dataset(root):
    root = Path(root)
    rows = []
    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue
        method = folder.name.lower()
        if method not in CLASS_NAMES:
            continue
        for p in sorted(folder.rglob('*')):
            if p.suffix.lower() in EXTENSIONS and not str(p).endswith('.txt'):
                rows.append({'path': str(p), 'method': method})
    df = pd.DataFrame(rows)
    le = LabelEncoder().fit(CLASS_NAMES)
    df['label'] = le.transform(df['method'])
    print(f"\n=== Dataset 2 — Classification technique ===")
    print(df['method'].value_counts().to_string())
    return df, le

df_classify, le = load_classification_dataset(CLASSIF_PATH)

print("\n✓ Datasets chargés !")

In [ ]:
# ═══════════════════════════════════════════════════════
# FEATURE EXTRACTORS — STAGE 1 
# ═══════════════════════════════════════════════════════

def to_gray(img):
    if img.ndim == 2:
        return img.astype(np.uint8)
    return cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2GRAY)

def load_image(path, size=(128, 128)):
    img = Image.open(path).convert('RGB').resize(size, Image.LANCZOS)
    return np.array(img, dtype=np.uint8)

# ── Filtres SRM ──────────────────────────────────────────
SRM_FILTERS = [
    np.array([[0,-1,0],[-1,4,-1],[0,-1,0]],    dtype=np.float32) / 4,
    np.array([[-1,2,-1],[2,-4,2],[-1,2,-1]],   dtype=np.float32) / 4,
    np.array([[0,0,0],[-1,2,-1],[0,0,0]],       dtype=np.float32) / 2,
    np.array([[0,-1,0],[0,2,0],[0,-1,0]],       dtype=np.float32) / 2,
    np.array([[-1,0,1],[-2,0,2],[-1,0,1]],     dtype=np.float32) / 8,
    np.array([[-1,-2,-1],[0,0,0],[1,2,1]],      dtype=np.float32) / 8,
    np.array([[1,-2,1],[-2,4,-2],[1,-2,1]],     dtype=np.float32) / 4,
    np.array([[0,1,0],[1,-4,1],[0,1,0]],        dtype=np.float32) / 4,
]

def extract_srm(gray):
    """8 filtres SRM × 9 bins = 72 dims"""
    feats = []
    g = gray.astype(np.float32)
    for f in SRM_FILTERS:
        res = cv2.filter2D(g, -1, f)
        res = np.clip(res, -4, 4)
        hist, _ = np.histogram(res, bins=9, range=(-4, 4), density=True)
        feats.append(hist.astype(np.float32))
    return np.concatenate(feats)  # 72 dims

def extract_lsb(gray):
    """Plans de bits 0-3 = 16 dims"""
    feats = []
    for bit in range(4):
        plane = (gray >> bit) & 1
        feats += [
            float(plane.mean()),
            float(np.abs(plane.mean() - 0.5)),
            float((plane[:-1,:] ^ plane[1:,:]).mean()),
            float((plane[:,:-1] ^ plane[:,1:]).mean()),
        ]
    return np.array(feats, dtype=np.float32)

def extract_dct_feats(gray):
    """Parité DCT blocs 8×8 = 5 dims"""
    h, w = gray.shape
    parities = []
    for r in range(0, h-7, 8):
        for c in range(0, w-7, 8):
            block = gray[r:r+8, c:c+8].astype(np.float64) - 128
            D = scipy_dct(scipy_dct(block.T, norm='ortho').T, norm='ortho')
            ac = D.flatten()[1:]
            parities.extend((np.round(ac) % 2).tolist())
    p = np.array(parities, dtype=np.float32) if parities else np.zeros(10, dtype=np.float32)
    mean_p = float(p.mean())
    return np.array([mean_p, float(p.std()),
                     float(np.abs(mean_p - 0.5)),
                     float((p==0).mean()), float((p==1).mean())],
                    dtype=np.float32)

def extract_fft_feats(gray):
    """Énergie par couronne fréquentielle = 8 dims"""
    F   = np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))
    mag = np.log1p(np.abs(F))
    h, w = mag.shape
    cy, cx = h//2, w//2
    Y, X = np.ogrid[:h, :w]
    R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(np.float32)
    max_r = min(cx, cy)
    feats = []
    for i in range(8):
        lo, hi = i*max_r/8, (i+1)*max_r/8
        mask = (R >= lo) & (R < hi)
        feats.append(float(mag[mask].mean()) if mask.any() else 0.0)
    return np.array(feats, dtype=np.float32)

def extract_stats(gray):
    """Statistiques globales = 8 dims"""
    g = gray.astype(np.float32).flatten()
    hist, _ = np.histogram(g, bins=256, range=(0,256), density=True)
    diff = np.diff(gray.astype(np.int16), axis=1).flatten()
    dh, _ = np.histogram(diff, bins=16, range=(-8,8), density=True)
    kurt_v = float(kurtosis(g)) if g.std() > 0 else 0.0
    skew_v = float(skew(g))     if g.std() > 0 else 0.0
    return np.array([
        float(g.mean()), float(g.std()), kurt_v, skew_v,
        float(hist[::32].mean()), float(dh.std()),
        float(np.abs(diff).mean()), float((np.abs(diff) < 2).mean())
    ], dtype=np.float32)

# ── Features file-format ─────────────────────────────────
B64_CHARS = set(b'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/=')
ZIP_MAGIC = b'PK\x03\x04'
JPEG_END  = b'\xff\xd9'
PNG_END   = b'IEND\xaeB`\x82'

def _byte_entropy(data):
    if not data:
        return 0.0
    counts = np.bincount(np.frombuffer(data, dtype=np.uint8), minlength=256)
    probs  = counts / counts.sum()
    probs  = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))

def format_features(path):
    """10 features file-format"""
    feats = np.zeros(10, dtype=np.float32)
    try:
        raw = open(str(path), 'rb').read()
    except:
        return feats
    feats[0] = float(np.log1p(len(raw)))
    try:
        with Image.open(path) as im:
            w, h = im.size
            feats[1] = float(len(raw) / max(w*h*len(im.getbands()), 1))
    except:
        pass
    try:
        with Image.open(path) as im:
            exif = im._getexif() if hasattr(im, '_getexif') else None
            if exif:
                feats[2] = 1.0
                feats[3] = float(np.log1p(len(str(exif))))
    except:
        pass
    ext = Path(str(path)).suffix.lower()
    appended = b''
    if ext in ('.jpg', '.jpeg'):
        idx = raw.rfind(JPEG_END)
        if idx != -1:
            appended = raw[idx+2:]
    elif ext == '.png':
        idx = raw.rfind(PNG_END)
        if idx != -1:
            appended = raw[idx+len(PNG_END):]
    feats[4] = float(np.log1p(len(appended)))
    if appended:
        feats[5] = _byte_entropy(appended)
        feats[6] = sum(32 <= b < 127 for b in appended) / len(appended)
        feats[7] = sum(b in B64_CHARS for b in appended) / len(appended)
        feats[8] = 1.0 if ZIP_MAGIC in appended else 0.0
    feats[9] = _byte_entropy(raw[-1024:] if len(raw) >= 1024 else raw)
    return feats

def pixel_features(path):
    """109 dims = SRM(72) + LSB(16) + DCT(5) + FFT(8) + Stats(8)"""
    gray = to_gray(load_image(path))
    return np.concatenate([
        extract_srm(gray),
        extract_lsb(gray),
        extract_dct_feats(gray),
        extract_fft_feats(gray),
        extract_stats(gray)
    ])

def features_stage1(path):
    """119 dims = pixel(109) + file-format(10)"""
    return np.concatenate([pixel_features(path), format_features(path)])

FEAT_S1 = 119
FEAT_S2 = 109

print("✓ Feature extractors prêts")
print(f"  Stage 1 : {FEAT_S1} dims (pixel=109 + format=10)")
print(f"  Stage 2 : {FEAT_S2} dims (pixel uniquement)")

In [ ]:
# ═══════════════════════════════════════════════════════
# EXTRACTION FEATURES STAGE 1
# ═══════════════════════════════════════════════════════
def extract_batch(df, feat_fn, desc, n_dims):
    out, errors = [], 0
    total = len(df)
    for i, path in enumerate(df['path']):
        try:
            out.append(feat_fn(path))
        except:
            out.append(np.zeros(n_dims, dtype=np.float32))
            errors += 1
        if i % 500 == 0:
            print(f"  {i}/{total}...")
    if errors:
        print(f"  ⚠ {errors} erreurs → remplacées par zéros")
    return np.array(out, dtype=np.float32)

# On limite à 2000 par classe pour équilibrer et aller plus vite
MAX = 2000
tr1_balanced = pd.concat([
    tr1[tr1['label']==0].head(MAX),
    tr1[tr1['label']==1].head(MAX)
]).reset_index(drop=True)

va1_balanced = pd.concat([
    va1[va1['label']==0].head(1000),
    va1[va1['label']==1].head(1000)
]).reset_index(drop=True)

te1_balanced = pd.concat([
    te1[te1['label']==0].head(1000),
    te1[te1['label']==1].head(1000)
]).reset_index(drop=True)

print(f"Train balancé : {len(tr1_balanced)} images")
print(f"Val balancé   : {len(va1_balanced)} images")
print(f"Test balancé  : {len(te1_balanced)} images")

print("\nExtraction features TRAIN...")
X1_train = extract_batch(tr1_balanced, features_stage1, 'Train', FEAT_S1)
y1_train = tr1_balanced['label'].values

print("\nExtraction features VAL...")
X1_val = extract_batch(va1_balanced, features_stage1, 'Val', FEAT_S1)
y1_val = va1_balanced['label'].values

print("\nExtraction features TEST...")
X1_test = extract_batch(te1_balanced, features_stage1, 'Test', FEAT_S1)
y1_test = te1_balanced['label'].values

# Imputation NaN
imputer1 = SimpleImputer(strategy='median')
X1_train = imputer1.fit_transform(X1_train)
X1_val   = imputer1.transform(X1_val)
X1_test  = imputer1.transform(X1_test)

print(f"\n✅ Features extraites !")
print(f"  Train : {X1_train.shape}")
print(f"  Val   : {X1_val.shape}")
print(f"  Test  : {X1_test.shape}")

In [ ]:
# ═══════════════════════════════════════════════════════
# ENTRAÎNEMENT STAGE 1 — Random Forest
# ═══════════════════════════════════════════════════════

# Poids de classe pour compenser le déséquilibre
cw1 = dict(enumerate(
    compute_class_weight('balanced',
                         classes=np.unique(y1_train),
                         y=y1_train)
))
print(f"Poids de classe : {{'clean': {cw1[0]:.3f}, 'stego': {cw1[1]:.3f}}}")

# Modèle Random Forest
detector = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        class_weight=cw1,
        max_features='sqrt',
        random_state=42
    ))
])

print("\nEntraînement Random Forest...")
detector.fit(X1_train, y1_train)
print("✓ Entraîné !")

# ── Optimisation du seuil ──────────────────────────────
y_prob_val = detector.predict_proba(X1_val)[:, 1]
thresholds = np.linspace(0.05, 0.95, 181)
rows_thr   = []

for t in thresholds:
    pred = (y_prob_val >= t).astype(int)
    cm   = confusion_matrix(y1_val, pred)
    cr   = cm[0,0] / cm[0].sum() if cm[0].sum() > 0 else 0
    sr   = cm[1,1] / cm[1].sum() if cm[1].sum() > 0 else 0
    f1v  = f1_score(y1_val, pred, zero_division=0)
    rows_thr.append({'threshold': t, 'f1': f1v,
                     'clean_rec': cr, 'stego_rec': sr})

df_thr = pd.DataFrame(rows_thr)

# Contrainte : clean recall ≥ 0.80
valid = df_thr[df_thr['clean_rec'] >= 0.80]
if len(valid) == 0:
    print("⚠ Contrainte relaxée")
    valid = df_thr

best_row  = valid.loc[valid['f1'].idxmax()]
THRESHOLD = float(best_row['threshold'])

print(f"\nSeuil optimal : {THRESHOLD:.2f}")
print(f"  F1           : {best_row['f1']:.4f}")
print(f"  Recall clean : {best_row['clean_rec']:.4f}")
print(f"  Recall stego : {best_row['stego_rec']:.4f}")

# ── Évaluation sur le test set ─────────────────────────
y1_prob = detector.predict_proba(X1_test)[:, 1]
y1_pred = (y1_prob >= THRESHOLD).astype(int)

print("\n" + "="*55)
print("  STAGE 1 — Résultats Test")
print("="*55)
print(f"  Accuracy : {accuracy_score(y1_test, y1_pred):.4f}")
print(f"  AUC-ROC  : {roc_auc_score(y1_test, y1_prob):.4f}")
print(f"  F1       : {f1_score(y1_test, y1_pred):.4f}")
print()
print(classification_report(y1_test, y1_pred,
      target_names=['Clean', 'Stego']))

# ── Matrice de confusion ───────────────────────────────
cm1 = confusion_matrix(y1_test, y1_pred)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Stage 1 — Détection binaire (Random Forest)', fontsize=13)

sns.heatmap(cm1, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=['Clean','Stego'],
            yticklabels=['Clean','Stego'])
axes[0].set_title('Matrice de confusion')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')

fpr, tpr, _ = roc_curve(y1_test, y1_prob)
auc_s1 = roc_auc_score(y1_test, y1_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC={auc_s1:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=0.8)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('Courbe ROC')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/stage1_results.png', dpi=100)
plt.show()

# ── Sauvegarder ───────────────────────────────────────
os.makedirs('/kaggle/working/models', exist_ok=True)
joblib.dump(detector,  '/kaggle/working/models/stage1_detector.pkl')
joblib.dump(imputer1,  '/kaggle/working/models/stage1_imputer.pkl')

meta_s1 = {
    'threshold' : THRESHOLD,
    'feat_dims' : FEAT_S1,
    'test_acc'  : round(float(accuracy_score(y1_test, y1_pred)), 4),
    'test_auc'  : round(float(roc_auc_score(y1_test, y1_prob)), 4),
    'test_f1'   : round(float(f1_score(y1_test, y1_pred)), 4),
}
with open('/kaggle/working/models/stage1_meta.json', 'w') as f:
    json.dump(meta_s1, f, indent=2)

print(f"\n✅ Stage 1 sauvegardé !")
print(f"  Seuil     : {THRESHOLD:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# MODÈLE 2 
# ═══════════════════════════════════════════════════════
IMG_SIZE    = 128
MAX_CLASS   = 1200  
BATCH_SIZE  = 32
NUM_CLASSES = len(CLASS_NAMES)

def extract_cnn_features_v2(img_array):
    original  = img_array
    img_255   = (img_array * 255).astype(np.uint8)
    lsb_plane = (img_255 % 2).astype(np.float32)
    smoothed  = uniform_filter(img_array, size=3)
    residual  = np.abs(img_array - smoothed)
    # Différence horizontale uniquement (pas verticale pour économiser RAM)
    diff_h = np.abs(np.diff(img_array, axis=1, prepend=img_array[:, :1, :]))
    return np.concatenate([
        original, lsb_plane, residual, diff_h
    ], axis=-1).astype(np.float32)  # 12 canaux

def load_folder_cnn_v2(folder_path, max_images):
    images = []
    all_files = [f for f in os.listdir(folder_path)
                 if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))
                 and not f.endswith('.txt')]
    np.random.shuffle(all_files)
    for filename in all_files[:max_images]:
        try:
            img = Image.open(os.path.join(folder_path, filename)).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            arr = np.array(img, dtype=np.float32) / 255.0
            images.append(extract_cnn_features_v2(arr))
        except:
            pass
    return images

all_images, all_labels = [], []
print("Chargement...\n")

for label_idx, class_name in enumerate(CLASS_NAMES):
    folder = f'{CLASSIF_PATH}/{class_name}'
    imgs   = load_folder_cnn_v2(folder, MAX_CLASS)
    all_images.extend(imgs)
    all_labels.extend([label_idx] * len(imgs))
    print(f'  {class_name}: {len(imgs)} images')

X = np.array(all_images, dtype=np.float32); del all_images
y = np.array(all_labels)
print(f'\nForme : {X.shape}')  # (N, 128, 128, 12)

# Découpage
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
del X
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
del X_temp

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f'  Train : {X_train.shape}')
print(f'  Val   : {X_val.shape}')
print(f'  Test  : {X_test.shape}')

# Augmentation
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    return image, label

train_ds2 = (tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
             .shuffle(500).batch(BATCH_SIZE)
             .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
             .prefetch(tf.data.AUTOTUNE))
val_ds2  = (tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_ds2 = (tf.data.Dataset.from_tensor_slices((X_test, y_test_cat))
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

# Architecture
model2_v2 = models.Sequential([
    layers.Input(shape=(128, 128, 12)),  # 12 canaux

    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='model2_v2')

model2_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_m2v2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/models/model2_v2_best.keras',
        monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, min_lr=0.0000001)
]

print("\nEntraînement...\n")
history2_v2 = model2_v2.fit(
    train_ds2, epochs=40,
    validation_data=val_ds2,
    callbacks=callbacks_m2v2
)
print("\n✅ Terminé !")

In [ ]:
# ═══════════════════════════════════════════════════════
# ÉVALUATION MODÈLE 2 V2
# ═══════════════════════════════════════════════════════
print("="*55)
print("  MODÈLE 2 V2 — Résultats Test")
print("="*55)

loss2, acc2 = model2_v2.evaluate(test_ds2, verbose=0)
print(f"\n  Test accuracy : {acc2:.4f}")
print(f"  Test loss     : {loss2:.4f}")

y_pred_v2 = np.argmax(model2_v2.predict(test_ds2, verbose=0), axis=1)

print(f"\nRapport détaillé :")
print(classification_report(y_test, y_pred_v2, target_names=CLASS_NAMES))

# Matrice de confusion
cm_v2 = confusion_matrix(y_test, y_pred_v2)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Modèle 2 V2 — Identification de technique', fontsize=13)

sns.heatmap(cm_v2, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('Matrice de confusion')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')
axes[0].tick_params(axis='x', rotation=45)

cm_v2_norm = cm_v2.astype(float) / cm_v2.sum(axis=1, keepdims=True)
sns.heatmap(cm_v2_norm, annot=True, fmt='.2f', ax=axes[1], cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, vmin=0, vmax=1)
axes[1].set_title('Matrice normalisée')
axes[1].set_xlabel('Prédit'); axes[1].set_ylabel('Réel')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/kaggle/working/model2_v2_results.png', dpi=100)
plt.show()

# Sauvegarder
model2_v2.save('/kaggle/working/models/model2_v2_best.keras')
print("\n✅ Modèle 2 V2 sauvegardé !")

In [ ]:
import tensorflow as tf

# Resauvegarder en format H5 (compatible avec toutes les versions)
model2_v2.save('/kaggle/working/models/model2_v2_best.h5')
print("✅ Modèle sauvegardé en format H5 !")

In [ ]:
# ── Test Modèle 2 uniquement ──────────────────────────
print("TEST MODÈLE 2 UNIQUEMENT\n")

CLASS_NAMES_M2 = ['dct', 'fft', 'lsb', 'lsb_grayscale', 'ssb4', 'ssbn']

for class_name, img_path in test_images.items():
    print(f"\n{'─'*50}")
    print(f"📌 Classe réelle : {class_name.upper()}")
    
    img         = Image.open(img_path).convert('RGB')
    img_resized = img.resize((128, 128), Image.NEAREST)
    img_array   = np.array(img_resized, dtype=np.float32) / 255.0
    features    = extract_cnn_features_v2(img_array)
    features    = np.expand_dims(features, axis=0)
    
    pred_m2   = model2_v2.predict(features, verbose=0)[0]
    tech_idx  = np.argmax(pred_m2)
    conf_m2   = pred_m2[tech_idx] * 100
    tech_name = CLASS_NAMES_M2[tech_idx]
    
    print(f"   Prédit    : {tech_name.upper()} ({conf_m2:.1f}%)")
    print(f"   Correct ? : {'✅' if tech_name == class_name else '❌'}")
    
    for name, prob in zip(CLASS_NAMES_M2, pred_m2):
        bar = '█' * int(prob * 20)
        print(f"   {name:<15} {bar:<20} {prob*100:.1f}%")

In [ ]:
# ── Pipeline final — Modèle 2 uniquement avec seuils ──
print("PIPELINE FINAL — MODÈLE 2 AVEC SEUILS\n")

SEUIL_LSB    = 0.80  # LSB doit être très confiant
SEUIL_AUTRES = 0.55  # autres techniques

def pipeline_final(img_path):
    print(f"\n{'='*55}")
    print(f"Analyse : {os.path.basename(img_path)}")
    print(f"{'='*55}")

    img         = Image.open(img_path).convert('RGB')
    img_resized = img.resize((128, 128), Image.NEAREST)
    img_array   = np.array(img_resized, dtype=np.float32) / 255.0
    features    = extract_cnn_features_v2(img_array)
    features    = np.expand_dims(features, axis=0)

    pred_m2   = model2_v2.predict(features, verbose=0)[0]
    tech_idx  = np.argmax(pred_m2)
    conf      = float(pred_m2[tech_idx])
    tech_name = CLASS_NAMES_M2[tech_idx]

    print(f"\n📊 Probabilités :")
    for name, prob in zip(CLASS_NAMES_M2, pred_m2):
        bar = '█' * int(prob * 25)
        print(f"   {name:<15} {bar:<25} {prob*100:.1f}%")

    # ── Décision finale ──────────────────────────────
    print(f"\n🎯 Décision :")

    if tech_name == 'lsb' and conf >= SEUIL_LSB:
        msg = decode_lsb_message(img_array)
        print(f"   🚨 STÉGANOGRAPHIE LSB DÉTECTÉE ({conf*100:.1f}%)")
        print(f"   💬 Message extrait : {msg[:100]}")
        return {'stego': True, 'method': 'LSB', 'message': msg}

    elif tech_name == 'lsb_grayscale' and conf >= SEUIL_LSB:
        msg = decode_lsb_message(img_array)
        print(f"   🚨 STÉGANOGRAPHIE LSB GRAYSCALE DÉTECTÉE ({conf*100:.1f}%)")
        print(f"   💬 Message extrait : {msg[:100]}")
        return {'stego': True, 'method': 'LSB_GRAYSCALE', 'message': msg}

    elif tech_name not in ['lsb', 'lsb_grayscale'] and conf >= SEUIL_AUTRES:
        print(f"   🚨 STÉGANOGRAPHIE {tech_name.upper()} DÉTECTÉE ({conf*100:.1f}%)")
        print(f"   💬 Extraction message non disponible pour {tech_name.upper()}")
        return {'stego': True, 'method': tech_name.upper(), 'message': None}

    else:
        print(f"   ✅ IMAGE POTENTIELLEMENT CLEAN")
        print(f"   (confiance max = {conf*100:.1f}% < seuil requis)")
        return {'stego': False, 'method': 'clean', 'message': None}

# ── Test ──────────────────────────────────────────────
for class_name, img_path in test_images.items():
    print(f"\n{'─'*55}")
    print(f"📌 Classe réelle : {class_name.upper()}")
    result = pipeline_final(img_path)

# 3rd Version : one model -----> Clean VS which type of stego

In [ ]:
import os
import numpy as np
from PIL import Image
from scipy.ndimage import uniform_filter
import tensorflow as tf
from keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

dataset_path     = '/kaggle/input/datasets/diegozanchett/digital-steganography'
clean_path_train = '/kaggle/input/datasets/marcozuppelli/stegoimagesdataset/train/train/clean'

IMG_SIZE    = 128
MAX_CLASS   = 1000
BATCH_SIZE  = 32

# 7 classes maintenant — clean inclus
CLASS_NAMES_FINAL = ['clean', 'dct', 'fft', 'lsb', 'lsb_grayscale', 'ssb4', 'ssbn']

def extract_cnn_features_v2(img_array):
    original  = img_array
    img_255   = (img_array * 255).astype(np.uint8)
    lsb_plane = (img_255 % 2).astype(np.float32)
    smoothed  = uniform_filter(img_array, size=3)
    residual  = np.abs(img_array - smoothed)
    diff_h    = np.abs(np.diff(img_array, axis=1,
                               prepend=img_array[:, :1, :]))
    return np.concatenate([
        original, lsb_plane, residual, diff_h
    ], axis=-1).astype(np.float32)

def load_folder_cnn(folder_path, max_images):
    images = []
    all_files = [f for f in os.listdir(folder_path)
                 if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))
                 and not f.endswith('.txt')]
    np.random.shuffle(all_files)
    for filename in all_files[:max_images]:
        try:
            img = Image.open(os.path.join(folder_path, filename)).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
            arr = np.array(img, dtype=np.float32) / 255.0
            images.append(extract_cnn_features_v2(arr))
        except:
            pass
    return images

# Dossiers pour chaque classe
folders = {
    'clean'        : clean_path_train,
    'dct'          : f'{dataset_path}/dct',
    'fft'          : f'{dataset_path}/fft',
    'lsb'          : f'{dataset_path}/lsb',
    'lsb_grayscale': f'{dataset_path}/lsb_grayscale',
    'ssb4'         : f'{dataset_path}/ssb4',
    'ssbn'         : f'{dataset_path}/ssbn',
}

all_images, all_labels = [], []
print("Chargement Dataset (7 classes)...\n")

for label_idx, class_name in enumerate(CLASS_NAMES_FINAL):
    folder = folders[class_name]
    imgs   = load_folder_cnn(folder, MAX_CLASS)
    all_images.extend(imgs)
    all_labels.extend([label_idx] * len(imgs))
    print(f'  {class_name} (label {label_idx}): {len(imgs)} images')

X = np.array(all_images, dtype=np.float32); del all_images
y = np.array(all_labels)

# Découpage 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
del X
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
del X_temp

NUM_CLASSES = len(CLASS_NAMES_FINAL)  # 7
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f'\n✅ Prêt !')
print(f'  Train : {X_train.shape}')
print(f'  Val   : {X_val.shape}')
print(f'  Test  : {X_test.shape}')
print(f'  Classes : {CLASS_NAMES_FINAL}')

# Data augmentation
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    return image, label

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
            .shuffle(1000).batch(BATCH_SIZE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds   = (tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_ds  = (tf.data.Dataset.from_tensor_slices((X_test, y_test_cat))
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

# Modèle CNN — 7 classes
model_final = models.Sequential([
    layers.Input(shape=(128, 128, 12)),

    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')  # 7 classes
], name='model_final_7classes')

model_final.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_final = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        '/kaggle/working/models/model_final_7classes.keras',
        monitor='val_accuracy', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, min_lr=0.0000001)
]

print("\nEntraînement Modèle Final (7 classes)...\n")
history_final = model_final.fit(
    train_ds, epochs=40,
    validation_data=val_ds,
    callbacks=callbacks_final
)
print("\n✅ Modèle Final terminé !")

In [ ]:
# ── Évaluation ────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("="*55)
print("  MODÈLE FINAL — 7 classes")
print("="*55)

loss, acc = model_final.evaluate(test_ds, verbose=0)
print(f"\n  Test accuracy : {acc:.4f}")
print(f"  Test loss     : {loss:.4f}")

y_pred = np.argmax(model_final.predict(test_ds, verbose=0), axis=1)
print(f"\nRapport détaillé :")
print(classification_report(y_test, y_pred,
      target_names=CLASS_NAMES_FINAL))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES_FINAL,
            yticklabels=CLASS_NAMES_FINAL)
plt.title('Modèle Final — 7 classes')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/kaggle/working/model_final_confusion.png', dpi=100)
plt.show()

# Sauvegarder
model_final.save('/kaggle/working/models/model_final_7classes.keras')
print("\n✅ Modèle final sauvegardé !")

In [ ]:
def decode_lsb_message(img_array):
    img_255 = (img_array * 255).astype(np.uint8).flatten()
    bits = [str(px & 1) for px in img_255]
    chars = []
    for i in range(0, len(bits) - 7, 8):
        byte = ''.join(bits[i:i+8])
        char = chr(int(byte, 2))
        if char == '\x00':  # null terminator = end of message
            break
        chars.append(char)
    return ''.join(chars)

In [ ]:
test_images = {
    'clean':         '/kaggle/input/datasets/marcozuppelli/stegoimagesdataset/train/train/clean/03000.png',
    'dct':           '/kaggle/input/datasets/diegozanchett/digital-steganography/dct/4000.bmp',
    'fft':           '/kaggle/input/datasets/diegozanchett/digital-steganography/fft/1023.bmp',
    'lsb':           '/kaggle/input/datasets/diegozanchett/digital-steganography/lsb/4023.png',
    'lsb_grayscale': '/kaggle/input/datasets/diegozanchett/digital-steganography/lsb_grayscale/1.png',
    'ssb4':          '/kaggle/input/datasets/diegozanchett/digital-steganography/ssb4/3023.png',
    'ssbn':          '/kaggle/input/datasets/diegozanchett/digital-steganography/ssbn/1009.png',
}
CLASS_NAMES_FINAL = ['clean', 'dct', 'fft', 'lsb', 'lsb_grayscale', 'ssb4', 'ssbn']

def pipeline_final_v2(img_path):
    print(f"\n{'='*55}")
    print(f"Analyse : {os.path.basename(img_path)}")
    print(f"{'='*55}")

    img         = Image.open(img_path).convert('RGB')  # ← fixed here
    img_resized = img.resize((128, 128), Image.NEAREST)
    img_array   = np.array(img_resized, dtype=np.float32) / 255.0
    features    = extract_cnn_features_v2(img_array)
    features    = np.expand_dims(features, axis=0)

    pred      = model_final.predict(features, verbose=0)[0]
    tech_idx  = np.argmax(pred)
    conf      = float(pred[tech_idx]) * 100
    tech_name = CLASS_NAMES_FINAL[tech_idx]

    print(f"\n📊 Probabilités :")
    for name, prob in zip(CLASS_NAMES_FINAL, pred):
        bar = '█' * int(prob * 25)
        print(f"   {name:<15} {bar:<25} {prob*100:.1f}%")

    print(f"\n🎯 Résultat :")
    if tech_name == 'clean':
        print(f"   ✅ IMAGE PROPRE — Aucune stéganographie ({conf:.1f}%)")
        return {'stego': False, 'method': 'clean'}
    elif tech_name in ['lsb', 'lsb_grayscale']:
        msg = decode_lsb_message(img_array)
        print(f"   🚨 STÉGANOGRAPHIE {tech_name.upper()} ({conf:.1f}%)")
        print(f"   💬 Message extrait : {msg[:100]}")
        return {'stego': True, 'method': tech_name, 'message': msg}
    else:
        print(f"   🚨 STÉGANOGRAPHIE {tech_name.upper()} ({conf:.1f}%)")
        print(f"   💬 Extraction non disponible pour {tech_name.upper()}")
        return {'stego': True, 'method': tech_name}

# ── Test sur toutes les classes ───────────────────────
print("TEST PIPELINE FINAL\n")

for class_name, img_path in test_images.items():
    print(f"\n{'─'*55}")
    print(f"📌 Classe réelle : {class_name.upper()}")
    result = pipeline_final_v2(img_path)

In [ ]:
# Sauvegarder en format H5 compatible
model_final.save('/kaggle/working/models/model_final_7classes.h5')
print("✅ Sauvegardé en H5 !")

# Vérifier la taille
import os
size = os.path.getsize('/kaggle/working/models/model_final_7classes.h5') / (1024*1024)
print(f"Taille : {size:.1f} MB")

In [ ]:
# ═══════════════════════════════════════════════════════
# LSB EMBEDDING
# ═══════════════════════════════════════════════════════
def embed_lsb_message(img_array, message):
    """
    Embed a text message into an image using LSB steganography.
    img_array : float32 array (H, W, 3) normalized [0,1]
    message   : string to hide
    returns   : stego image as float32 array [0,1]
    """
    img_255 = (img_array * 255).astype(np.uint8).copy()
    
    # Add null terminator to mark end of message
    message += '\x00'
    
    # Convert message to bits
    bits = []
    for char in message:
        byte = format(ord(char), '08b')
        bits.extend([int(b) for b in byte])
    
    flat = img_255.flatten()
    
    if len(bits) > len(flat):
        raise ValueError(f"Message too long! Max {len(flat)//8 - 1} characters for this image.")
    
    # Embed each bit into the LSB of each pixel byte
    for i, bit in enumerate(bits):
        flat[i] = (flat[i] & 0xFE) | bit  # clear LSB then set it
    
    stego = flat.reshape(img_255.shape).astype(np.float32) / 255.0
    return stego


def decode_lsb_message(img_array):
    """
    Extract a hidden LSB message from an image.
    img_array : float32 array (H, W, 3) normalized [0,1]
    returns   : decoded string
    """
    img_255 = (img_array * 255).astype(np.uint8).flatten()
    bits = [str(px & 1) for px in img_255]
    chars = []
    for i in range(0, len(bits) - 7, 8):
        byte = ''.join(bits[i:i+8])
        char = chr(int(byte, 2))
        if char == '\x00':  # null terminator = end of message
            break
        chars.append(char)
    return ''.join(chars)



In [ ]:
test_images['lsb_test'] = '/kaggle/working/stego_lsb_full.png'
result = pipeline_final_v2('/kaggle/working/stego_lsb_full.png')

In [ ]:
# ── Embed a message that fills the entire image ───────
img_full      = Image.open(test_images['clean']).convert('RGB')
img_array_full = np.array(img_full, dtype=np.float32) / 255.0

# Max capacity = H * W * 3 bytes, each holds 1 bit
# So max message length in chars = (H * W * 3) // 8 - 1
h, w, c       = img_array_full.shape
max_chars     = (h * w * c) // 8 - 1
print(f"Image size       : {h}x{w}x{c}")
print(f"Max message size : {max_chars} characters")

# Fill ~80% of capacity to guarantee heavy modification
secret = "A" * int(max_chars * 0.80)
print(f"Embedding        : {len(secret)} characters ({len(secret)/max_chars*100:.0f}% of capacity)")

stego_lsb_full = embed_lsb_message(img_array_full, secret)

# Verify
orig_255  = (img_array_full * 255).astype(np.uint8)
stego_255 = (stego_lsb_full * 255).astype(np.uint8)
diff      = np.abs(orig_255.astype(int) - stego_255.astype(int))
print(f"\nPixels changed   : {(diff > 0).sum()} / {diff.size} ({(diff > 0).mean()*100:.1f}%)")

# Save and test
stego_pil = Image.fromarray(stego_255)
stego_pil.save('/kaggle/working/stego_lsb_heavy.png')
print("\n✅ Saved — now testing through pipeline...\n")

result = pipeline_final_v2('/kaggle/working/stego_lsb_heavy.png')